# TAT-DQA Dataset Exploration & Layout Audit

This notebook explores the **TAT-DQA (Tabular And Textual Document Question Answering)** benchmark corpus used in **AuditRAG**.

### Objectives:
1. Load and inspect the hierarchical document layout trees.
2. Analyze corpus statistics (document count, page distributions, question density).
3. Inspect question answer types (arithmetic derivations vs extractive spans).
4. Examine layout bounding-box coordinates for visual citation grounding.


In [1]:
from pathlib import Path
import json
import pandas as pd
from PIL import Image

from auditrag.config import PROJECT_ROOT, RAW_DATA_DIR, RAW_TEST_GOLD_JSON
from auditrag.data.loader import TATDQALoader

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")


## 1. Load Dataset Corpus
We use  to ingest the gold test benchmark split.


In [2]:
loader = TATDQALoader()
dataset = loader.load_dataset(use_gold=True, load_doc_details=False)

print(f"Total unique documents: {len(dataset.documents)}")
print(f"Total benchmark questions: {len(dataset.questions)}")
print(f"Average questions per document: {len(dataset.questions) / max(len(dataset.documents), 1):.1f}")


## 2. Question Types & Derivation Breakdown
TAT-DQA questions require both extractive span retrieval and deterministic arithmetic reasoning.


In [3]:
q_types = [q.answer_type for q in dataset.questions]
df_qtypes = pd.Series(q_types).value_counts().reset_index()
df_qtypes.columns = ["Answer Type", "Count"]
df_qtypes["Percentage"] = (df_qtypes["Count"] / len(q_types) * 100).round(1).astype(str) + "%"
df_qtypes


## 3. Sample Arithmetic Question & Derivation
Demonstrating a question requiring arithmetic reasoning (e.g. percentage change or variance).


In [4]:
arith_questions = [q for q in dataset.questions if q.answer_type == "arithmetic"]
sample_q = arith_questions[0] if arith_questions else dataset.questions[0]

print(f"Question: {sample_q.question}")
print(f"Answer: {sample_q.answer}")
print(f"Derivation formula: {sample_q.derivation}")
print(f"Scale: {sample_q.scale}")
print(f"Target Document UID: {sample_q.doc_uid}")


## 4. Document Layout Hierarchy & Coordinate Space
Inspect the structural OCR blocks and bounding box coordinates for evidence grounding.


In [5]:
sample_doc = dataset.get_document(sample_q.doc_uid)
if sample_doc and sample_doc.pages:
    p1 = sample_doc.pages[0]
    print(f"Page 1 dimensions: {p1.width} x {p1.height} px")
    print(f"Total layout blocks on page: {len(p1.blocks)}")
    print(f"Sample block 1 bbox: {p1.blocks[0].bbox if p1.blocks else None}")
    print(f"Sample block 1 text: {p1.blocks[0].text[:80] if p1.blocks else None}...")
else:
    print("Document details not loaded in memory; read raw JSON directly.")
